# Task 3: Heart Disease Prediction

**Internship:** DevelopersHub Corporation – AI/ML Engineering  
**Objective:** Build a binary classification model to predict the presence of heart disease using the UCI Heart Disease dataset.  
**Dataset:** UCI Heart Disease Dataset (Cleveland)  
**Models:** Logistic Regression + Decision Tree (GridSearchCV)

## Problem Statement

Heart disease is the number one cause of death worldwide. The purpose of this task is to build a machine learning model that can predict whether a person has heart disease or not based on their medical data. Early detection can save lives by enabling timely medical intervention.

## Goal

- Load and inspect the Heart Disease UCI dataset.
- Clean the dataset and handle missing values.
- Perform Exploratory Data Analysis (EDA) to understand trends.
- Preprocess features using StandardScaler.
- Train Logistic Regression and Decision Tree models.
- Evaluate using accuracy, confusion matrix, and ROC curve.
- Highlight the most important features affecting prediction.

## Importing Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score
)

sns.set_style('whitegrid')

## Loading the Dataset

The Heart Disease UCI dataset is loaded directly from a public GitHub URL.  
No manual download needed — just run the cells.

**Backup:** If URL fails, download `heart.csv` from:  
https://www.kaggle.com/datasets/ronitf/heart-disease-uci  
Then use: `df = pd.read_csv('heart.csv')`

In [ ]:
url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/refs/heads/master/heart_disease.csv'

try:
    df = pd.read_csv(url)
    print("Dataset loaded successfully.")
except Exception as e:
    print(f"URL failed: {e}")
    print("Please place heart.csv in this folder.")
    raise

print("\nDataset shape:", df.shape)
df.head()

## Dataset Inspection

Checking dataset shape, columns and basic information.

In [ ]:
print("Column Names:", df.columns.tolist())
print("\nDataset Info:")
df.info()

## Data Preprocessing

Checking for missing values and cleaning the dataset.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

df.dropna(inplace=True)
print(f"\nAfter dropping missing values: {df.shape}")

# Target is already binary (0 = no disease, 1 = disease)
print("\nTarget distribution:")
print(df['target'].value_counts())

## Statistical Summary

In [ ]:
df.describe()

## EDA — Target Distribution

Visualizing how many patients have heart disease vs those who do not.

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x='target', data=df)
plt.title('Distribution of Heart Disease (0=No, 1=Yes)')
plt.xlabel('Target')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## EDA — Correlation Heatmap

Understanding relationships between all features in the dataset.

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## EDA — Pairplot

Pairplot of selected features to see distribution and relationships colored by target.

In [ ]:
selected = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'target']
sns.pairplot(df[selected], hue='target', diag_kind='kde')
plt.suptitle('Pairplot of Selected Features', y=1.02)
plt.tight_layout()
plt.show()

## EDA — Boxplots

Boxplots show distribution of each feature by target class and help detect outliers.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
for i, feat in enumerate(features):
    row, col = i // 3, i % 3
    sns.boxplot(x='target', y=feat, data=df, ax=axes[row, col])
    axes[row, col].set_title(f'{feat} vs target')
axes[1, 2].set_visible(False)
plt.tight_layout()
plt.show()

## Feature and Target Split + Scaling

Splitting into 80% training and 20% testing. StandardScaler applied for Logistic Regression.

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Training set shape:", X_train_scaled.shape)
print("Test set shape:    ", X_test_scaled.shape)

## Model Training — Logistic Regression

In [ ]:
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
print("Logistic Regression trained successfully.")

## Model Training — Decision Tree with GridSearchCV

In [ ]:
param_grid = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
dt = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(dt, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_scaled, y_train)
best_dt = grid_search.best_estimator_

print("Decision Tree trained successfully.")
print("Best parameters:", grid_search.best_params_)

## Model Evaluation — Accuracy & Classification Report

In [ ]:
y_pred_log = log_reg.predict(X_test_scaled)
y_pred_dt  = best_dt.predict(X_test_scaled)

acc_log = accuracy_score(y_test, y_pred_log)
acc_dt  = accuracy_score(y_test, y_pred_dt)

print(f"Logistic Regression Accuracy: {acc_log * 100:.2f}%")
print(f"Decision Tree Accuracy:       {acc_dt  * 100:.2f}%")

print("\n=== Logistic Regression Classification Report ===")
print(classification_report(y_test, y_pred_log,
      target_names=['No Disease', 'Disease']))

print("\n=== Decision Tree Classification Report ===")
print(classification_report(y_test, y_pred_dt,
      target_names=['No Disease', 'Disease']))

## Model Evaluation — Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(confusion_matrix(y_test, y_pred_log), annot=True, fmt='d',
            cmap='Blues', ax=axes[0],
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
axes[0].set_title('Logistic Regression — Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(confusion_matrix(y_test, y_pred_dt), annot=True, fmt='d',
            cmap='Greens', ax=axes[1],
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
axes[1].set_title('Decision Tree — Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## Model Evaluation — ROC Curve

In [ ]:
y_proba_log = log_reg.predict_proba(X_test_scaled)[:, 1]
y_proba_dt  = best_dt.predict_proba(X_test_scaled)[:, 1]

fpr_log, tpr_log, _ = roc_curve(y_test, y_proba_log)
fpr_dt,  tpr_dt,  _ = roc_curve(y_test, y_proba_dt)
auc_log = roc_auc_score(y_test, y_proba_log)
auc_dt  = roc_auc_score(y_test, y_proba_dt)

plt.figure(figsize=(8, 6))
plt.plot(fpr_log, tpr_log, label=f'Logistic Regression (AUC = {auc_log:.2f})', color='blue')
plt.plot(fpr_dt,  tpr_dt,  label=f'Decision Tree       (AUC = {auc_dt:.2f})',  color='green')
plt.plot([0, 1], [0, 1], 'k--', label='Random Baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Logistic Regression vs Decision Tree')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Logistic Regression AUC: {auc_log:.2f}")
print(f"Decision Tree AUC:       {auc_dt:.2f}")

## Feature Importance Analysis

In [ ]:
coef_log = pd.Series(
    np.abs(log_reg.coef_[0]), index=X.columns
).sort_values(ascending=False)

print("Logistic Regression Feature Importance:")
print(coef_log)

importances_dt = pd.Series(
    best_dt.feature_importances_, index=X.columns
).sort_values(ascending=False)

print("\nDecision Tree Feature Importances:")
print(importances_dt)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

coef_log.plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Logistic Regression Coefficients (Absolute)')
axes[0].set_xlabel('Features')
axes[0].set_ylabel('Coefficient Value')
axes[0].tick_params(axis='x', rotation=45)

importances_dt.plot(kind='bar', ax=axes[1], color='lightgreen')
axes[1].set_title('Decision Tree Feature Importances')
axes[1].set_xlabel('Features')
axes[1].set_ylabel('Importance Score')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Decision Tree Structure
plt.figure(figsize=(20, 10))
plot_tree(best_dt, feature_names=X.columns.tolist(),
          class_names=['No Disease', 'Disease'],
          filled=True, max_depth=3)
plt.title('Decision Tree Structure (max_depth=3)')
plt.tight_layout()
plt.show()

# Final Insights

### Observations

1. **Logistic Regression** achieves **~80.33% accuracy** and AUC of **0.87** — strong performance.
2. **Decision Tree** achieves **~75.41% accuracy** and AUC of **0.82** — good but slightly lower.
3. **Top important features:** cp (chest pain type), thal, oldpeak, ca, thalach.
4. **Confusion matrix** shows Logistic Regression handles both classes better.
5. **ROC curves** confirm Logistic Regression outperforms Decision Tree on this dataset.
6. Dataset is **clean and balanced** — 165 disease vs 138 non-disease cases.

### Conclusion

Logistic Regression is the better model here — simpler, more interpretable, and higher AUC. Chest pain type (cp), blood disorder type (thal), and ST depression (oldpeak) are the most informative features. Further improvements can be made using Random Forest, XGBoost, or deep learning approaches.